## 5. LightGBM OOF

- **타겟**: 원단위 `Calories_Burned` 직접 학습 (log 변환 X)  
  → 이 데이터는 다항식 구조에 가까워 log 변환이 오히려 역효과  
- **num_leaves=63**: 7500샘플에 맞게 조정 (과적합 방지)  
- **early stopping**: 200 라운드 자동 종료  
- **categorical_feature**: Gender, Weight_Status 지정

## 1. 라이브러리 · 시드 · 상수

In [1]:
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.model_selection import KFold
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler, LabelEncoder
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

RANDOM_STATE = 42
N_SPLITS = 5
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

## 2. 데이터 로드

In [2]:
train = pd.read_csv("train.csv")
test  = pd.read_csv("test.csv")
submission = pd.read_csv("sample_submission.csv")

print("Train:", train.shape, "  Test:", test.shape)
print(train.head(2))

Train: (7500, 11)   Test: (7500, 10)
           ID  Exercise_Duration  Body_Temperature(F)    BPM  Height(Feet)  \
0  TRAIN_0000               26.0                105.6  107.0           5.0   
1  TRAIN_0001                7.0                103.3   88.0           6.0   

   Height(Remainder_Inches)  Weight(lb)  Weight_Status Gender  Age  \
0                       9.0       154.3  Normal Weight      F   45   
1                       6.0       224.9     Overweight      M   50   

   Calories_Burned  
0            166.0  
1             33.0  


## 3. Feature Engineering — 수정금지

### 추가된 물리 공식 피처
| 피처 | 공식 | 근거 |
|------|------|------|
| `Keyser_total` | 성별별 Keyser(2001) 공식 × Duration | 심박수 기반 칼로리 직접 추정 |
| `HRR_pct` | (BPM - RestingHR) / (MaxHR - RestingHR) | 운동 강도 상대적 지표 |
| `MaxHR` | 208 - 0.7 × Age (Tanaka 공식) | 나이 보정된 최대 심박수 |
| `BMR` | Harris-Benedict 공식 (성별 분리) | 기초대사량 기준선 |
| `Log_Keyser` | log1p(max(Keyser_total, 0)) | 음수 방지 + 스케일 안정화 |

In [3]:
def create_features_extended(df):
    df = df.copy()

    # ─── 기존 피처 (ver7 동일) ───────────────────────────────
    df['Height_Total_Inches'] = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Temp_diff'] = df['Body_Temperature(F)'] - 98.6

    df['Duration_bin'] = pd.cut(
        df['Exercise_Duration'], bins=[-np.inf, 10, 20, 30, np.inf], labels=[0, 1, 2, 3]
    ).astype(int)

    df['Duration_x_BPM']     = df['Exercise_Duration'] * df['BPM']
    df['Duration_x_TempDiff']= df['Exercise_Duration'] * df['Temp_diff']
    df['BPM_x_TempDiff']     = df['BPM'] * df['Temp_diff']
    df['Intensity']          = df['Duration_x_BPM']
    df['Effort']             = df['Weight(lb)'] * df['Intensity']

    df['Duration_sq']        = df['Exercise_Duration'] ** 2
    df['Temp_diff_sq']       = df['Temp_diff'] ** 2
    df['Dur_BPM_TempDiff']   = df['Exercise_Duration'] * df['BPM'] * df['Temp_diff']
    df['BPM_per_Duration']   = df['BPM'] / (df['Exercise_Duration'] + 1)
    df['TempDiff_per_Duration'] = df['Temp_diff'] / (df['Exercise_Duration'] + 1)

    h2 = (df['Height_Total_Inches'] ** 2).replace(0, np.nan)
    df['BMI'] = 703 * df['Weight(lb)'] / h2
    df['BMI'] = df['BMI'].fillna(df['BMI'].median())

    df['Weight_x_Duration']    = df['Weight(lb)'] * df['Exercise_Duration']
    df['Log_BPM']              = np.log1p(df['BPM'])
    df['Log_Duration']         = np.log1p(df['Exercise_Duration'])
    df['Log_Weight_BPM_Dur']   = np.log1p(df['Weight(lb)'] * df['BPM'] * df['Exercise_Duration'])

    # ─── 신규: 물리 공식 기반 피처 ───────────────────────────
    df['Weight_kg']  = df['Weight(lb)'] * 0.453592
    df['Height_cm']  = df['Height_Total_Inches'] * 2.54
    is_male          = (df['Gender'] == 'M').astype(float)

    # Tanaka 최대 심박수 공식 (2001): 나이 보정
    df['MaxHR']    = 208 - 0.7 * df['Age']
    # 심박수 예비량 % — 운동 강도의 상대적 지표 (안정 HR=60 가정)
    df['HRR_pct']  = ((df['BPM'] - 60) / (df['MaxHR'] - 60)).clip(0, 1)

    # Keyser (2001) 심박수 기반 칼로리/분 공식
    keyser_m = (-55.0969 + 0.6309 * df['BPM'] + 0.1988 * df['Weight_kg'] + 0.2017 * df['Age']) / 4.184
    keyser_f = (-20.4022 + 0.4472 * df['BPM'] - 0.1263 * df['Weight_kg'] + 0.0740 * df['Age']) / 4.184
    df['Keyser_per_min'] = is_male * keyser_m + (1 - is_male) * keyser_f
    df['Keyser_total']   = df['Keyser_per_min'] * df['Exercise_Duration']   # 총 칼로리 추정
    df['Log_Keyser']     = np.log1p(np.clip(df['Keyser_total'], 0, None))   # 음수 방지

    # Harris-Benedict BMR (기초대사량) — 성별 분리
    bmr_m = 88.362  + 13.397 * df['Weight_kg'] + 4.799 * df['Height_cm'] - 5.677 * df['Age']
    bmr_f = 447.593 +  9.247 * df['Weight_kg'] + 3.098 * df['Height_cm'] - 4.330 * df['Age']
    df['BMR']           = is_male * bmr_m + (1 - is_male) * bmr_f
    df['BMR_per_min']   = df['BMR'] / (24 * 60)                            # 분당 BMR

    # 나이 × 운동 관련 상호작용
    df['Age_x_BPM']      = df['Age'] * df['BPM']
    df['Age_x_Duration'] = df['Age'] * df['Exercise_Duration']
    df['Age_x_Effort']   = df['Age'] * df['Effort']

    # Keyser vs 실제 관계를 모델이 학습하도록 비율 피처
    df['Effort_per_Keyser'] = df['Effort'] / (df['Keyser_total'].abs() + 1)

    return df

# 확인
sample = create_features_extended(train.head(3))
new_cols = [c for c in sample.columns if c not in train.columns]
print(f"신규 피처 수: {len(new_cols)}")
print("신규 피처:", new_cols)
print()
print(sample[['Keyser_total', 'Log_Keyser', 'BMR', 'HRR_pct', 'MaxHR']].to_string())

신규 피처 수: 31
신규 피처: ['Height_Total_Inches', 'Temp_diff', 'Duration_bin', 'Duration_x_BPM', 'Duration_x_TempDiff', 'BPM_x_TempDiff', 'Intensity', 'Effort', 'Duration_sq', 'Temp_diff_sq', 'Dur_BPM_TempDiff', 'BPM_per_Duration', 'TempDiff_per_Duration', 'BMI', 'Weight_x_Duration', 'Log_BPM', 'Log_Duration', 'Log_Weight_BPM_Dur', 'Weight_kg', 'Height_cm', 'MaxHR', 'HRR_pct', 'Keyser_per_min', 'Keyser_total', 'Log_Keyser', 'BMR', 'BMR_per_min', 'Age_x_BPM', 'Age_x_Duration', 'Age_x_Effort', 'Effort_per_Keyser']

   Keyser_total  Log_Keyser          BMR   HRR_pct  MaxHR
0    136.329473    4.922383  1442.889034  0.403433  176.5
1     51.508645    3.960978  2121.955908  0.247788  173.0
2     41.315398    3.745151  2164.497833  0.203602  187.7


## 4. 전처리

- **LightGBM용**: Label Encoding (Gender, Weight_Status) — LGBM이 수치형으로 처리
- **Ridge용**: OHE → base + TOP5 pruning (ver7 구조 유지)

In [4]:
# ── 공통: raw split ──────────────────────────────────────────
train_x_raw = train.drop(['ID', 'Calories_Burned'], axis=1).copy()
train_y     = train['Calories_Burned'].astype(float).copy()
test_x_raw  = test.drop(['ID'], axis=1).copy()

y     = train_y.values

# ── Feature Engineering ───────────────────────────────────────
before_cols = list(train_x_raw.columns)

train_feat = create_features_extended(train_x_raw)
test_feat  = create_features_extended(test_x_raw)

generated_cols = [c for c in train_feat.columns if c not in before_cols]
cat_cols = ['Gender', 'Weight_Status']

# ── (A) LightGBM용 데이터 — Label Encoding ────────────────────
train_lgb = train_feat.copy()
test_lgb  = test_feat.copy()

for c in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train_lgb[c], test_lgb[c]], axis=0))
    train_lgb[c] = le.transform(train_lgb[c])
    test_lgb[c]  = le.transform(test_lgb[c])

lgb_cat_cols = cat_cols  # LGBM에 categorical 알려줄 컬럼 목록

# ── (B) Ridge용 데이터 — OHE + pruning (ver7 동일) ───────────
enc = OneHotEncoder(drop='first', handle_unknown='ignore', sparse_output=False)
enc.fit(train_feat[cat_cols])

tr_cat = enc.transform(train_feat[cat_cols])
te_cat = enc.transform(test_feat[cat_cols])
cat_names = enc.get_feature_names_out(cat_cols)

train_ohe = train_feat.drop(columns=cat_cols).copy()
test_ohe  = test_feat.drop(columns=cat_cols).copy()
train_ohe[cat_names] = tr_cat
test_ohe[cat_names]  = te_cat

# Ridge 피처 pruning: base + TOP5
TOP5_FIXED = ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']
base_feats = [c for c in train_ohe.columns if c not in generated_cols]
keep_cols  = base_feats + TOP5_FIXED
train_ridge = train_ohe[keep_cols].copy()
test_ridge  = test_ohe[keep_cols].copy()

print(f"LightGBM 피처 수  : {train_lgb.shape[1]}")
print(f"Ridge 피처 수     : {train_ridge.shape[1]}")
print(f"TOP5 present      : {[c for c in TOP5_FIXED if c in train_ridge.columns]}")

LightGBM 피처 수  : 40
Ridge 피처 수     : 15
TOP5 present      : ['Log_Weight_BPM_Dur', 'Effort', 'Weight_x_Duration', 'Log_Duration', 'Intensity']


## 5. LightGBM OOF

- **타겟**: `log1p(Calories_Burned)` → 예측 후 `expm1()` 복원  
- **early stopping**: 200 라운드 (과적합 자동 방지)  
- **categorical_feature**: Gender, Weight_Status 지정

In [5]:
lgb_params = {
    'objective'        : 'regression',      # MSE 최소화 (원단위 직접 학습)
    'metric'           : 'rmse',
    'learning_rate'    : 0.02,
    'num_leaves'       : 63,               # 7500샘플에 맞게 127→63으로 줄임
    'max_depth'        : -1,
    'min_child_samples': 20,
    'feature_fraction' : 0.8,
    'bagging_fraction' : 0.8,
    'bagging_freq'     : 5,
    'lambda_l1'        : 0.1,
    'lambda_l2'        : 1.0,
    'verbose'          : -1,
    'random_state'     : RANDOM_STATE,
}
N_EST    = 3000
EARLY_STOP = 200

lgb_oof  = np.zeros(len(train_lgb))
lgb_test = np.zeros(len(test_lgb))
best_rounds = []

for fold, (tr_idx, va_idx) in enumerate(kf.split(train_lgb)):
    X_tr, X_va = train_lgb.iloc[tr_idx], train_lgb.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]   # ★ log 변환 없이 원단위 그대로

    dtrain = lgb.Dataset(X_tr, label=y_tr, categorical_feature=lgb_cat_cols, free_raw_data=False)
    dvalid = lgb.Dataset(X_va, label=y_va, reference=dtrain, free_raw_data=False)

    model = lgb.train(
        lgb_params, dtrain,
        num_boost_round=N_EST,
        valid_sets=[dvalid],
        callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False),
                   lgb.log_evaluation(500)]
    )

    lgb_oof[va_idx] = model.predict(X_va)          # ★ expm1 없음
    lgb_test        += model.predict(test_lgb) / N_SPLITS
    best_rounds.append(model.best_iteration)

    fold_rmse = rmse(y_va, np.clip(lgb_oof[va_idx], 0, None))
    print(f"  Fold {fold+1} | best_iter={model.best_iteration:4d} | RMSE={fold_rmse:.5f}")

lgb_oof  = np.clip(lgb_oof,  0, None)
lgb_test = np.clip(lgb_test, 0, None)

print(f"\n[LightGBM] OOF RMSE : {rmse(y, lgb_oof):.5f}")
print(f"           평균 best_round: {int(np.mean(best_rounds))}")

[500]	valid_0's rmse: 1.38551
[1000]	valid_0's rmse: 1.30141
[1500]	valid_0's rmse: 1.27608
[2000]	valid_0's rmse: 1.2648
[2500]	valid_0's rmse: 1.25984
[3000]	valid_0's rmse: 1.2567
  Fold 1 | best_iter=3000 | RMSE=1.25670
[500]	valid_0's rmse: 1.38382
[1000]	valid_0's rmse: 1.31017
[1500]	valid_0's rmse: 1.28602
[2000]	valid_0's rmse: 1.27638
[2500]	valid_0's rmse: 1.26972
[3000]	valid_0's rmse: 1.26495
  Fold 2 | best_iter=3000 | RMSE=1.26495
[500]	valid_0's rmse: 1.51236
[1000]	valid_0's rmse: 1.42863
[1500]	valid_0's rmse: 1.42191
[2000]	valid_0's rmse: 1.41977
  Fold 3 | best_iter=1865 | RMSE=1.41744
[500]	valid_0's rmse: 1.64845
[1000]	valid_0's rmse: 1.52975
[1500]	valid_0's rmse: 1.4951
[2000]	valid_0's rmse: 1.47988
[2500]	valid_0's rmse: 1.4713
[3000]	valid_0's rmse: 1.46624
  Fold 4 | best_iter=3000 | RMSE=1.46624
[500]	valid_0's rmse: 1.47276
[1000]	valid_0's rmse: 1.3547
[1500]	valid_0's rmse: 1.31402
[2000]	valid_0's rmse: 1.2962
[2500]	valid_0's rmse: 1.28224
[3000]	val

## 6. LightGBM 피처 중요도 (Keyser 공식 효과 확인)

In [6]:
# 마지막 fold 모델로 중요도 확인 (참고용)
imp_df = pd.DataFrame({
    'feature'   : train_lgb.columns,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=False)

print(imp_df.head(20).to_string(index=False))

          feature   importance
   Duration_x_BPM 3.376350e+08
        Intensity 9.624123e+07
 Dur_BPM_TempDiff 1.333465e+07
          HRR_pct 1.172952e+07
     Keyser_total 1.071550e+07
       Log_Keyser 2.151593e+06
     Age_x_Effort 9.282698e+05
   Age_x_Duration 7.899361e+05
        Age_x_BPM 6.365263e+05
   Keyser_per_min 4.489930e+05
              BMR 3.721596e+05
           Gender 2.613794e+05
          Log_BPM 1.336953e+05
Exercise_Duration 1.216527e+05
              BPM 1.128876e+05
      BMR_per_min 7.787184e+04
 BPM_per_Duration 7.157357e+04
       Weight(lb) 4.898999e+04
Effort_per_Keyser 4.836341e+04
   BPM_x_TempDiff 3.698232e+04


## 7. Ridge OOF (ver7 구조 — 앙상블 베이스)

> Ridge는 LightGBM이 놓치는 **선형 구조**를 보완합니다.

In [7]:
DEGREE   = 3
ALPHA_ID = 0.05
ALPHA_SQ = 0.009

y_sqrt = np.sqrt(np.clip(y, 0, None))

id_oof  = np.zeros(len(train_ridge));  id_test  = np.zeros(len(test_ridge))
sq_oof  = np.zeros(len(train_ridge));  sq_test  = np.zeros(len(test_ridge))

for fold, (tr_idx, va_idx) in enumerate(kf.split(train_ridge)):
    X_tr_r, X_va_r = train_ridge.iloc[tr_idx], train_ridge.iloc[va_idx]

    poly = PolynomialFeatures(degree=DEGREE, include_bias=False)
    X_tr = poly.fit_transform(X_tr_r)
    X_va = poly.transform(X_va_r)
    X_te = poly.transform(test_ridge)

    sc = StandardScaler()
    X_tr_s = sc.fit_transform(X_tr)
    X_va_s = sc.transform(X_va)
    X_te_s = sc.transform(X_te)

    # Identity Ridge
    m_id = Ridge(alpha=ALPHA_ID, random_state=RANDOM_STATE)
    m_id.fit(X_tr_s, y[tr_idx])
    id_oof[va_idx] = m_id.predict(X_va_s)
    id_test += m_id.predict(X_te_s) / N_SPLITS

    # Sqrt Ridge
    m_sq = Ridge(alpha=ALPHA_SQ, random_state=RANDOM_STATE)
    m_sq.fit(X_tr_s, y_sqrt[tr_idx])
    pred_va_sq = m_sq.predict(X_va_s);  pred_te_sq = m_sq.predict(X_te_s)
    sq_oof[va_idx] = np.clip(pred_va_sq, 0, None) ** 2
    sq_test += (np.clip(pred_te_sq, 0, None) ** 2) / N_SPLITS

id_oof  = np.clip(id_oof,  0, None);  sq_oof  = np.clip(sq_oof,  0, None)
id_test = np.clip(id_test, 0, None);  sq_test = np.clip(sq_test, 0, None)

# Ridge 내부 최적 가중치
def obj_ridge(w):
    return rmse(y, w * id_oof + (1 - w) * sq_oof)

res_r    = minimize(lambda w: obj_ridge(w[0]), x0=[0.99], bounds=[(0, 1)])
W_ID_r   = float(res_r.x[0])
ridge_oof  = np.clip(W_ID_r * id_oof  + (1 - W_ID_r) * sq_oof,  0, None)
ridge_test = np.clip(W_ID_r * id_test + (1 - W_ID_r) * sq_test, 0, None)

print(f"[Ridge] w_identity={W_ID_r:.4f} | OOF RMSE: {rmse(y, ridge_oof):.5f}")

[Ridge] w_identity=0.9852 | OOF RMSE: 0.29937


## 8. 최종 앙상블 — LightGBM + Ridge 최적 가중치 결합

scipy 최적화로 LightGBM:Ridge 최적 비율 자동 탐색

In [8]:
def obj_ensemble(w):
    # w = LightGBM 비율
    pred = np.clip(w * lgb_oof + (1 - w) * ridge_oof, 0, None)
    return rmse(y, pred)

res_ens = minimize(lambda w: obj_ensemble(w[0]), x0=[0.7], bounds=[(0, 1)])
W_LGB   = float(res_ens.x[0])
W_RIDGE = 1 - W_LGB

final_oof  = np.clip(W_LGB * lgb_oof  + W_RIDGE * ridge_oof,  0, None)
final_test = np.clip(W_LGB * lgb_test + W_RIDGE * ridge_test, 0, None)

print("=" * 55)
print(f"  LightGBM  단독 OOF RMSE : {rmse(y, lgb_oof):.5f}")
print(f"  Ridge     단독 OOF RMSE : {rmse(y, ridge_oof):.5f}")
print(f"  앙상블    OOF RMSE      : {rmse(y, final_oof):.5f}")
print("=" * 55)
print(f"  최적 가중치 — LightGBM : {W_LGB:.4f} | Ridge : {W_RIDGE:.4f}")
print(f"  Test 예측 mean/min/max  : {final_test.mean():.2f} / {final_test.min():.2f} / {final_test.max():.2f}")

  LightGBM  단독 OOF RMSE : 1.33906
  Ridge     단독 OOF RMSE : 0.29937
  앙상블    OOF RMSE      : 0.29937
  최적 가중치 — LightGBM : 0.0000 | Ridge : 1.0000
  Test 예측 mean/min/max  : 89.70 / 0.57 / 313.35


## 9. 제출 파일 저장

In [9]:
oof_rmse = rmse(y, final_oof)
fname    = f"submit_lgb_keyser_{oof_rmse:.5f}.csv"

submission["Calories_Burned"] = final_test
submission.to_csv(fname, index=False)
print(f"저장 완료: {fname}")

저장 완료: submit_lgb_keyser_0.29937.csv
